# Introspective advantage in pretrained language models (GPT-2 and Pythia)

Companion experiment to *Where Does Introspective Advantage Come From?* (Caro and Toscano, JAIR).
It measures, for small pretrained language models, the type-2 sensitivity of three monitors:

* **public**: sees only the prompt;
* **behaviour (q samples)**: sees the greedy answer and `q` sampled answers, spread over the prompt and two paraphrases;
* **white box**: also reads the logits and a 32-dimensional projection of the last hidden state.

The trace level cannot be measured here, because the training data are not available per item.
Monitors are trained on half of the items and evaluated on the other half (20 random splits).

Runtime: about 30–60 minutes on a free Colab GPU (Runtime > Change runtime type > T4). It also runs on CPU, more slowly.
At the end it writes `lm_pretrained_summary.json`; send that file back to integrate the results in the paper.

In [ ]:
!pip -q install transformers==4.44.2 accelerate scikit-learn

In [ ]:
import json, re, random, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
DEV = "cuda" if torch.cuda.is_available() else "cpu"
MODELS = ["gpt2", "gpt2-medium", "EleutherAI/pythia-160m", "EleutherAI/pythia-410m"]
Q_LEVELS = [0, 3, 12, 48]
SEED = 0; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(DEV)

## Items: capitals (factual recall) and two-digit addition (computation)

In [ ]:
CAPITALS = {"France":"Paris","Germany":"Berlin","Italy":"Rome","Spain":"Madrid","Portugal":"Lisbon","Japan":"Tokyo",
"China":"Beijing","India":"New Delhi","Russia":"Moscow","Canada":"Ottawa","Mexico":"Mexico City","Brazil":"Brasilia",
"Argentina":"Buenos Aires","Chile":"Santiago","Peru":"Lima","Colombia":"Bogota","Venezuela":"Caracas","Ecuador":"Quito",
"Bolivia":"La Paz","Uruguay":"Montevideo","Paraguay":"Asuncion","Cuba":"Havana","Egypt":"Cairo","Kenya":"Nairobi",
"Nigeria":"Abuja","Ghana":"Accra","Ethiopia":"Addis Ababa","Morocco":"Rabat","Algeria":"Algiers","Tunisia":"Tunis",
"South Africa":"Pretoria","Australia":"Canberra","New Zealand":"Wellington","Indonesia":"Jakarta","Thailand":"Bangkok",
"Vietnam":"Hanoi","Philippines":"Manila","Malaysia":"Kuala Lumpur","Pakistan":"Islamabad","Bangladesh":"Dhaka",
"Iran":"Tehran","Iraq":"Baghdad","Syria":"Damascus","Turkey":"Ankara","Greece":"Athens","Poland":"Warsaw",
"Ukraine":"Kyiv","Sweden":"Stockholm","Norway":"Oslo","Finland":"Helsinki","Denmark":"Copenhagen","Ireland":"Dublin",
"Netherlands":"Amsterdam","Belgium":"Brussels","Austria":"Vienna","Switzerland":"Bern","Hungary":"Budapest",
"Czech Republic":"Prague","Romania":"Bucharest","Bulgaria":"Sofia","Serbia":"Belgrade","Croatia":"Zagreb",
"Slovakia":"Bratislava","Slovenia":"Ljubljana","Lithuania":"Vilnius","Latvia":"Riga","Estonia":"Tallinn",
"Iceland":"Reykjavik","Israel":"Jerusalem","Jordan":"Amman","Lebanon":"Beirut","Saudi Arabia":"Riyadh",
"Qatar":"Doha","Kuwait":"Kuwait City","Oman":"Muscat","Yemen":"Sanaa","Afghanistan":"Kabul","Nepal":"Kathmandu",
"Sri Lanka":"Colombo","Mongolia":"Ulaanbaatar","North Korea":"Pyongyang","South Korea":"Seoul","Taiwan":"Taipei",
"Kazakhstan":"Astana","Uzbekistan":"Tashkent","Georgia":"Tbilisi","Armenia":"Yerevan","Azerbaijan":"Baku",
"Belarus":"Minsk","Moldova":"Chisinau","Albania":"Tirana","Cyprus":"Nicosia","Malta":"Valletta","Luxembourg":"Luxembourg",
"Senegal":"Dakar","Mali":"Bamako","Niger":"Niamey","Chad":"N'Djamena","Sudan":"Khartoum","Uganda":"Kampala",
"Tanzania":"Dodoma","Rwanda":"Kigali","Zambia":"Lusaka","Zimbabwe":"Harare","Angola":"Luanda","Mozambique":"Maputo",
"Madagascar":"Antananarivo","Namibia":"Windhoek","Botswana":"Gaborone","Cameroon":"Yaounde","Libya":"Tripoli",
"Panama":"Panama City","Costa Rica":"San Jose","Nicaragua":"Managua","Honduras":"Tegucigalpa","Guatemala":"Guatemala City",
"El Salvador":"San Salvador","Jamaica":"Kingston","Haiti":"Port-au-Prince","Dominican Republic":"Santo Domingo"}
CAP_T = ["The capital of {c} is", "{c}'s capital city is", "Q: What is the capital of {c}? A:"]
rng = np.random.default_rng(0)
ADD = [(int(a), int(b)) for a, b in rng.integers(10, 100, size=(150, 2))]
ADD_T = ["{a} + {b} =", "The sum of {a} and {b} is", "Q: What is {a} plus {b}? A:"]
def items(task):
    if task == "facts":
        return [dict(prompts=[t.format(c=c) for t in CAP_T], gold=g) for c, g in CAPITALS.items()]
    return [dict(prompts=[t.format(a=a, b=b) for t in ADD_T], gold=str(a + b)) for a, b in ADD]
def norm(s):
    s = s.strip().split("\n")[0]
    s = re.split(r"[.,;!?]", s)[0]
    return s.strip().lower()
def correct(ans, gold):
    a, g = norm(ans), gold.lower()
    return a.startswith(g) or (len(a) > 0 and g.startswith(a) and len(a) >= 3)

## Features for one model

In [ ]:
@torch.no_grad()
def run_model(name, task, max_new=6):
    tok = AutoTokenizer.from_pretrained(name); tok.pad_token = tok.eos_token; tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(name).to(DEV).eval()
    its = items(task); rows = []
    for it in its:
        p0 = it["prompts"][0]
        enc = tok(p0, return_tensors="pt").to(DEV)
        out = model(**enc, output_hidden_states=True)
        logits = out.logits[0, -1].float(); pr = torch.softmax(logits, -1)
        top2 = torch.topk(logits, 2).values
        h = out.hidden_states[-1][0, -1].float().cpu().numpy()
        greedy = model.generate(**enc, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        ans = tok.decode(greedy[0, enc.input_ids.shape[1]:])
        # sampled answers: prompt and paraphrases, Q_max samples spread round-robin
        qmax = max(Q_LEVELS); per = qmax // 3 + 1; samples = []
        for j, p in enumerate(it["prompts"]):
            e = tok([p] * per, return_tensors="pt", padding=True).to(DEV)
            g = model.generate(**e, max_new_tokens=max_new, do_sample=True, temperature=1.0, pad_token_id=tok.eos_token_id)
            samples.append([norm(tok.decode(x[e.input_ids.shape[1]:])) for x in g])
        order = [(k % 3, k // 3) for k in range(qmax)]
        seq = [samples[j][i] for j, i in order]
        rows.append(dict(Z=int(correct(ans, it["gold"])), ans=norm(ans), seq=seq, order=[j for j, _ in order],
                         plen=len(p0), maxp=float(pr.max()), margin=float(top2[0] - top2[1]),
                         ent=float(-(pr * (pr + 1e-12).log()).sum()), hnorm=float(np.linalg.norm(h)), h=h))
    return rows

def build(rows):
    n = len(rows); Z = np.array([r["Z"] for r in rows])
    pub = np.array([[r["plen"]] for r in rows], float)
    F = {"public": pub}
    for q in Q_LEVELS:
        if q == 0:
            F["beh_q0"] = pub; continue
        cols = []
        for r in rows:
            s = np.array(r["seq"][:q]); o = np.array(r["order"][:q])
            own = (s[o == 0] == r["ans"]).mean() if (o == 0).any() else -1
            par = (s[o != 0] == r["ans"]).mean() if (o != 0).any() else -1
            cols.append([own, par, len(set(s.tolist())) / q])
        F[f"beh_q{q}"] = np.column_stack([pub, np.array(cols)])
    H = np.stack([r["h"] for r in rows])
    wb = np.array([[r["maxp"], r["margin"], r["ent"], r["hnorm"]] for r in rows])
    F["white"] = np.column_stack([F[f"beh_q{max(Q_LEVELS)}"], wb])
    return Z, F, H

## Cross-validated monitors

In [ ]:
def evaluate(Z, F, H, splits=20):
    res = {k: [] for k in F}
    res["white"] = []
    for s in range(splits):
        r = np.random.default_rng(s); idx = r.permutation(len(Z)); a, b = idx[: len(Z) // 2], idx[len(Z) // 2:]
        if Z[a].min() == Z[a].max() or Z[b].min() == Z[b].max():
            continue
        pca = PCA(n_components=min(32, len(a) - 1), random_state=s).fit(H[a])
        for k, X in F.items():
            if k == "white":
                X = np.column_stack([X, pca.transform(H)])
            m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=7,
                                               l2_regularization=1.0, random_state=s).fit(X[a], Z[a])
            res[k].append(roc_auc_score(Z[b], m.predict_proba(X[b])[:, 1]))
    return {k: np.array(v) for k, v in res.items()}

summary = []
for name in MODELS:
    for task in ["facts", "arithmetic"]:
        rows = run_model(name, task)
        Z, F, H = build(rows)
        res = evaluate(Z, F, H)
        line = dict(model=name, task=task, acc=float(Z.mean()), n=int(len(Z)))
        for k, v in res.items():
            line[k] = float(np.mean(v)) if len(v) else None
        for q in Q_LEVELS:
            d = res["white"] - res[f"beh_q{q}"] if len(res["white"]) else np.array([])
            line[f"adv_q{q}"] = float(d.mean()) if len(d) else None
            line[f"adv_q{q}_sd"] = float(d.std()) if len(d) else None
        summary.append(line); print(line, flush=True)
json.dump(summary, open("lm_pretrained_summary.json", "w"), indent=1)

Download `lm_pretrained_summary.json` from the Files panel on the left and send it back.